In [60]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [61]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score

In [62]:
df=pd.read_csv("insurance.csv")

In [63]:
df.head(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
0,67,119.8,1.56,2.92,False,Jaipur,retired,High
1,36,101.1,1.83,34.28,False,Chennai,freelancer,Low
2,39,56.8,1.64,36.64,False,Indore,freelancer,Low
3,22,109.4,1.55,3.34,True,Mumbai,student,Medium
4,69,62.2,1.60,3.94,True,Indore,retired,High


In [64]:
df['occupation'].unique()

<StringArray>
[       'retired',     'freelancer',        'student', 'government_job',
 'business_owner',     'unemployed',    'private_job']
Length: 7, dtype: str

In [65]:
df_feat=df.copy()

In [66]:
df_feat['bmi']=df_feat['weight']/(df_feat['height']**2)


In [67]:
def age_group(age:int)->str:
    if age<25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
    else:
        return "senior"
    
    

In [68]:
df_feat['age_group']=df_feat['age'].apply(age_group)

In [69]:
def lifestyle_risk(row):#row is pandas series object
    if row['smoker'] and row['bmi']>30:
        return "high"
    elif row['smoker'] or row['bmi']>30:
        return "medium"
    else:
        return "low"

In [70]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [71]:
def classify_city(city:str)->str:
    if city in tier_1_cities:
        return "Tier-1"
    elif city in tier_2_cities:
        return "Tier-2"
    else:
        return "Tier-3"


In [72]:
df_feat['city_tier']=df_feat['city'].apply(classify_city)

In [73]:
df_feat.head()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi,age_group,city_tier
0,67,119.8,1.56,2.92,False,Jaipur,retired,High,49.227482,senior,Tier-2
1,36,101.1,1.83,34.28,False,Chennai,freelancer,Low,30.189017,adult,Tier-1
2,39,56.8,1.64,36.64,False,Indore,freelancer,Low,21.118382,adult,Tier-2
3,22,109.4,1.55,3.34,True,Mumbai,student,Medium,45.535900,young,Tier-1
4,69,62.2,1.60,3.94,True,Indore,retired,High,24.296875,senior,Tier-2


In [74]:
df_feat['lifestyle_risk'] = df_feat.apply(lifestyle_risk, axis=1)

df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[
    ['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']
].sample(5)
     # first we drop the unnecessary columns and then we reorder the columns for better visulization
     #income->occupation->bmi->age_group->lifestyle_risk->city_tier->target

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
45,18.390000,unemployed,33.466667,middle_aged,medium,Tier-2,High
44,50.000000,private_job,30.078125,middle_aged,high,Tier-2,Medium
56,2.860000,student,42.414152,young,high,Tier-1,Medium
14,13.505166,government_job,32.800735,middle_aged,medium,Tier-3,Medium
43,1.560000,retired,29.308163,senior,low,Tier-1,Medium


In [75]:
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
y = df_feat["insurance_premium_category"]

In [76]:
X

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,49.227482,senior,medium,Tier-2,2.92000,retired
1,30.189017,adult,medium,Tier-1,34.28000,freelancer
2,21.118382,adult,low,Tier-2,36.64000,freelancer
3,45.535900,young,high,Tier-1,3.34000,student
4,24.296875,senior,medium,Tier-2,3.94000,retired
...,...,...,...,...,...,...
95,21.420747,adult,low,Tier-2,19.64000,business_owner
96,47.984483,adult,medium,Tier-1,34.01000,private_job
97,18.765432,middle_aged,low,Tier-1,44.86000,freelancer
98,30.521676,adult,medium,Tier-1,28.30000,business_owner


In [77]:
y


0       High
1        Low
2        Low
3     Medium
4       High
       ...  
95       Low
96       Low
97       Low
98       Low
99       Low
Name: insurance_premium_category, Length: 100, dtype: str

In [78]:
# Define categorical and numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]# apply one hot encoding to the categorical features 
numeric_features = ["bmi", "income_lpa"]# for numerical features we can ignore

In [79]:
preprocessor=ColumnTransformer(# apply different transformations to different columns
    transformers=[
        ('cat', OneHotEncoder(), categorical_features),
        ('num', "passthrough", numeric_features)
    ]
)

In [80]:
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression




In [81]:
RF=RandomForestClassifier(random_state=42)
GB=GradientBoostingClassifier(random_state=42)
estimators=[('rf', RF), ('gb', GB)]
stacking_clf=StackingClassifier(estimators=estimators, final_estimator=RandomForestClassifier(random_state=42))

In [82]:
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', "passthrough", numeric_features)
    ]
)

In [83]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.4,random_state=42)

In [84]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [85]:
from sklearn.linear_model import LogisticRegression

stacking_clf = StackingClassifier(
    estimators=estimators, 
    final_estimator=LogisticRegression()
)

In [86]:
y_pred=pipeline.predict(X_test)

In [87]:
accuracy_score(y_test,y_pred)

0.725

In [88]:
import pickle 
pickle_model_path="model.pkl"
with open(pickle_model_path, 'wb') as f:# save the model to a pickle file name model.pkl
    pickle.dump(pipeline,f)